In [ ]:
import sys
from pytorch_lightning.loggers import CometLogger

# sys.path.append(".")
sys.path.append("..")

from pathlib import Path
import numpy as np
import pandas as pd
from lib.wpt_utils import load_wavelets
from training.feature_processing import (
    new_extract_features,
    build_left_right_transition_matrix,
)
from training.objects import ScalerParams, NewSpectralFeatures
from training.constants import SEQ_1_PATH, SEQ_2_PATH, FS, CABLES
from training.dataset import (
    build_wpt_dataset_from_features,
    new_build_tensor_stacked_dataset,
)
from torch.utils.data import TensorDataset, DataLoader
from training.dataset import normalize_tensor
from training.model import VAEClassifier
import pytorch_lightning as pl
import torch
from training.evaluate import evaluate_vae, evaluate_hmm
from hmmlearn.hmm import GaussianHMM
from training.train_vae_hmm_pipe import vae_hmm_pipeline


In [ ]:
# -- Load data --
data_valid = load_wavelets(".."/ SEQ_1_PATH)
data_train = load_wavelets(".."/ SEQ_2_PATH)

In [ ]:
fixed_params = {
    "num_bins": 10,
    "normalize": True,
    "batch_size": 128,
    "num_epochs": 200,
    "lr": 1e-3,
    "latent_dim": 2,
    "hidden_dim": 16,
    "num_layers": 2,
    "activation": "relu",
    "hmm_components": 6,
    "max_level": 7,
}

variable_params = {
    "f_min": 10,
    "bandwidth": 12,
    "beta": 0.1,
}

params = {**fixed_params, **variable_params}
f_max = params["f_min"] + params["bandwidth"]

In [ ]:
output = vae_hmm_pipeline(data_train, data_valid, params)